In [ ]:
import os
import re
import arviz as az
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.lines import Line2D

# Load and Clean Survey Data

In [ ]:
raw_df = pd.read_csv("../data/survey_data/analysis.csv")

main_mapping = pd.read_csv("../data/survey_data/survey_codebook.csv", sep="\t")
options_mapping = pd.read_csv("../data/survey_data/options_codebook.csv", sep="\t")

display(main_mapping.iloc[3]["Label"])
display(options_mapping)

In [ ]:
raw_df.columns

In [ ]:
def parse_question_text(label):
    """Extract the common question text and specific option from a Q1a-style label"""
    # Find the variable part (e.g., "Q1a_1") at the beginning
    var_match = re.match(r"(Q\d+[a-z])(_\d+)", label)
    if not var_match:
        return None, None

    # Get Q1a (without the _1, _2, etc.)
    base_question_id = var_match.group(1)  # This will be "Q1a"
    rest_of_label = label[len(var_match.group(0)) :].strip()

    # Split on the question mark to separate question from option
    parts = rest_of_label.split("?")
    if len(parts) >= 2:
        question_text = base_question_id + " " + parts[0].strip() + "?"
        option_text = parts[1].strip()
        return question_text, option_text

    return None, None

In [ ]:
def create_question_mapping(main_mapping, options_mapping):
    """Create a comprehensive question mapping dictionary"""
    question_mapping = {}

    # Process options_mapping to create proper value mappings
    option_maps = {}
    current_variable = None
    option_counter = 1

    for _, row in options_mapping.iterrows():
        value = row["Value"]
        label = row["Label"]

        # If Value is not NaN, it's a new variable with its first option
        if pd.notna(value):
            current_variable = value
            option_counter = 1
            option_maps[current_variable] = {}
            # The first option is on the same row as the variable name
            option_maps[current_variable][str(option_counter)] = label
            option_counter += 1

        # If Value is NaN, it's a continuation option for the current variable
        elif current_variable is not None and pd.isna(value):
            option_maps[current_variable][str(option_counter)] = label
            option_counter += 1

    # Process all variables from main_mapping
    for _, row in main_mapping.iterrows():
        var = row["Variable"]
        label = row["Label"]

        # Extract question ID (e.g., Q1a from Q1a_1)
        question_id_match = re.match(r"([A-Za-z0-9]+)", var)
        if question_id_match:
            question_id = question_id_match.group(1)

            if question_id not in question_mapping:
                question_mapping[question_id] = {
                    "question_text": None,
                    "question_options": {},
                    "option_mapping": None,
                }

            # Check if this is a Q1a-style question with options
            if re.match(r"Q\d+[a-z]_\d+", var):
                question_text, option_text = parse_question_text(label)

                if question_text and option_text:
                    # Extract option number
                    option_num = var.split("_")[-1]

                    # Update question text if not already set
                    if question_mapping[question_id]["question_text"] is None:
                        question_mapping[question_id]["question_text"] = question_text

                    # Add option
                    question_mapping[question_id]["question_options"][
                        option_num
                    ] = option_text

            else:
                # For variables not in Q1a_N format (like age, ethnicity, etc.)
                question_mapping[question_id]["question_text"] = label

                # Check if this variable has options in options_mapping
                if var in option_maps:
                    question_mapping[question_id]["option_mapping"] = option_maps[var]

    return question_mapping

In [ ]:
def create_structured_dataframe(raw_df, question_mapping):
    """Create a structured dataframe with proper question text and options"""
    structured_data = raw_df.copy()

    # Add metadata columns
    structured_data.columns.name = "variables"

    # Create a metadata dictionary for columns
    column_metadata = {}

    for col in structured_data.columns:
        col_key = col

        # Find the question ID (e.g., Q1a from Q1a_1)
        question_id_match = re.match(r"([A-Za-z0-9]+)", col)
        if question_id_match:
            question_id = question_id_match.group(1)

            if question_id in question_mapping:
                col_metadata = {
                    "variable": col,
                    "question_text": question_mapping[question_id]["question_text"],
                    "question_options": None,
                    "option_mapping": question_mapping[question_id]["option_mapping"],
                }

                # For Q1a_N style columns, get the specific option
                if re.match(r"Q\d+[a-z]_\d+", col):
                    option_num = col.split("_")[-1]
                    if option_num in question_mapping[question_id]["question_options"]:
                        col_metadata["specific_option"] = question_mapping[question_id][
                            "question_options"
                        ][option_num]

                column_metadata[col] = col_metadata

    # Create a summary dataframe with question information
    questions_df = pd.DataFrame.from_dict(column_metadata, orient="index")

    return structured_data, questions_df

In [ ]:
# Main processing
question_mapping = create_question_mapping(main_mapping, options_mapping)
structured_df, questions_df = create_structured_dataframe(raw_df, question_mapping)

# Example usage - display question info
print("Question Metadata Sample:")
for q_id, q_info in list(question_mapping.items())[:3]:
    print(f"\n{q_id}:")
    print(f"  Question: {q_info['question_text']}")
    if q_info["question_options"]:
        print(f"  Options: {q_info['question_options']}")
    if q_info["option_mapping"]:
        print(f"  Option Mapping: {q_info['option_mapping']}")

# Display sample of structured data
print("\nStructured Data Sample:")
print(structured_df.head())

print("\nQuestions DataFrame Sample:")
print(questions_df.head())

In [ ]:
question2option = {}
for i, row in questions_df.iterrows():
    variable = row["variable"]
    specific_option = row["specific_option"]
    option_mapping = row["option_mapping"]
    if pd.isna(specific_option) == False:
        question2option[variable] = specific_option
    if pd.isna(option_mapping) == False:
        question2option[variable] = option_mapping

In [ ]:
print(questions_df["question_text"].unique())

In [ ]:
def get_weighted_selected_counts(
    question_text, question_id, structured_df, question2option
):
    """
    Replicate MATLAB weighting approach exactly
    """
    ppt_id_col = "ID"
    weight_col = "W8"  # Weight column name

    # Get relevant variables (like MATLAB)
    relevant_vars = [
        col for col in structured_df.columns if col.startswith(question_id)
    ]

    # Include ID, weight, and relevant variables
    df = structured_df[[ppt_id_col, weight_col] + relevant_vars].copy()

    # Remove "_other" columns as in your original code
    remove_cols = [col for col in df.columns if "_other" in col]
    if remove_cols:
        df = df.drop(columns=remove_cols)

    # Rename columns using the mapping
    rename_dict = {k: v for k, v in question2option.items() if k in df.columns}
    df = df.rename(columns=rename_dict)

    # Handle empty strings (convert to NaN) but DON'T drop rows yet
    df = df.replace(r"^\s*$", np.nan, regex=True)

    # Get total respondents (like MATLAB n_respondents = 2499)
    total_respondents = len(df)

    # Convert response columns to numeric (excluding ID and weight)
    response_cols = [col for col in df.columns if col not in [ppt_id_col, weight_col]]
    for col in response_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Apply weighting exactly like MATLAB
    weighted_counts = {}
    for col in response_cols:
        # Only include responses where value = 1 and weight is valid
        valid_mask = (df[col] == 1) & df[weight_col].notna()

        # Sum the weights for selected responses (MATLAB approach)
        if valid_mask.any():
            weighted_count = df.loc[valid_mask, weight_col].sum()
            weighted_counts[col] = round(weighted_count)
        else:
            weighted_counts[col] = 0

    # Convert to Series and sort
    option_counts = pd.Series(weighted_counts).sort_values(ascending=False)

    # Calculate weighted respondents who answered this question set
    # Sum of weights for people who have at least one valid response
    any_response_mask = df[response_cols].notna().any(axis=1) & df[weight_col].notna()
    weighted_respondents = round(df.loc[any_response_mask, weight_col].sum())

    return option_counts, total_respondents, weighted_respondents


def get_weighted_option_counts(
    question_text, question_id, structured_df, question2option
):
    """
    For single-answer questions (like Q1c - frequency of chatbot use)
    """
    ppt_id_col = "ID"
    weight_col = "W8"

    # Get the specific question column
    relevant_cols = [ppt_id_col, weight_col, question_id]
    df = structured_df[relevant_cols].copy()

    # Replace answer codes with labels if mapping exists
    if question_id in question2option and isinstance(
        question2option[question_id], dict
    ):
        df[question_id] = df[question_id].replace(question2option[question_id])

    # Handle empty strings
    df = df.replace(r"^\s*$", np.nan, regex=True)

    total_respondents = len(df)

    # Remove rows where question response or weight is missing
    valid_df = df.dropna(subset=[question_id, weight_col])

    # Calculate weighted counts for each option
    weighted_counts = {}
    for option in valid_df[question_id].unique():
        option_mask = valid_df[question_id] == option
        weighted_count = valid_df.loc[option_mask, weight_col].sum()
        weighted_counts[option] = round(weighted_count)

    # Convert to Series and sort
    option_counts = pd.Series(weighted_counts).sort_values(ascending=False)

    # Weighted respondents who answered this question
    weighted_respondents = round(valid_df[weight_col].sum())

    return option_counts, total_respondents, weighted_respondents


def plot_weighted_hbar(
    option_counts, total_respondents, weighted_respondents, question_text
):
    """
    Plot with weighted counts and proper denominators
    """
    plt.figure(figsize=(10, 3))

    # Calculate percentages using total respondents (MATLAB approach)
    percentages = (option_counts / weighted_respondents * 100).round(1)
    # percentages = (option_counts / total_respondents * 100).round(1)

    df_plot = pd.DataFrame(
        {
            "option": option_counts.index,
            "count": option_counts.values,
            "percentage": percentages.values,
        }
    )

    sns.barplot(
        data=df_plot,
        x="count",
        y="option",
        hue="option",
        palette="viridis",
        legend=False,
    )

    # Add percentage labels on the bars (your original style)
    for i, (count, percentage) in enumerate(
        zip(df_plot["count"], df_plot["percentage"])
    ):
        plt.text(count + 5, i, f"{percentage}%", va="center", ha="left", fontsize=10)

    # Show weighted respondents in title (updated from your original)
    plt.title(f"N = {weighted_respondents}", fontsize=12)
    plt.annotate(
        f"{question_text}",
        xy=(0.0, 1.2),
        xycoords="axes fraction",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold",
    )

    # Keep your original styling
    plt.xlabel("")
    plt.ylabel("")
    plt.xlim(0, max(option_counts.values) * 1.2)
    plt.tight_layout()
    plt.show()

### Methods

In [ ]:
question_text = "Q1a Over the past four weeks, which of the following have you used find out about UK political issues or current affairs?"
question_id = "Q1a"
option_counts, total_respondents, weighted_respondents = get_weighted_selected_counts(
    question_text, question_id, structured_df, question2option
)
plot_weighted_hbar(
    option_counts, total_respondents, weighted_respondents, question_text
)

# Keep only Internet Search and AI Chatbot columns
option_counts = option_counts[
    ["Internet Search (for example Google)", "AI chatbots (for example ChatGPT)"]
]
print(option_counts)
survey_uses = option_counts.copy()

print(f"Weighted respondents for Q1a: {weighted_respondents}")

In [ ]:
question_text = "Q1d Which, if any, of the following AI chatbots have you used in the last four weeks?"
question_id = "Q1d"
option_counts, total_respondents, weighted_respondents = get_weighted_selected_counts(
    question_text, question_id, structured_df, question2option
)
plot_weighted_hbar(
    option_counts, total_respondents, weighted_respondents, question_text
)

print(f"Weighted respondents for Q1d: {weighted_respondents}")

In [ ]:
question_text = "Q1e In the last four weeks, have you asked an AI chatbot for information or advice on any of the following topics?"
question_id = "Q1e"
option_counts, total_respondents, weighted_respondents = get_weighted_selected_counts(
    question_text, question_id, structured_df, question2option
)
plot_weighted_hbar(
    option_counts, total_respondents, weighted_respondents, question_text
)
print(option_counts)
survey_usecases = option_counts.copy()
total_users_usecases = weighted_respondents

print(f"Weighted respondents for Q1e: {weighted_respondents}")

In [ ]:
question_bank = {
    "Q5b": "Q5b How useful, if at all, were the AI chatbot’s replies?",
    "Q5c": "Q5c How accurate, if at all, did the AI chatbot’s replies seem?",
    "Q5e": "Q5e Did the AI chatbot’s replies seem to be fair and balanced, or did the replies favour left-wing views over right-wing views, or vice versa?",
    "Q5f": "Q5f Did the way the AI chatbot replied influence your perspective on the issues that you researched?",
    "Q3d": "Q3d Thinking about the information  about UK current affairs or political issues that was generated by an AI chatbot, did it make you more likely to vote, less likely to vote, or did it make no difference?",
}
for question_id, question_text in question_bank.items():
    option_counts, total_users_orig, weighted_respondents = get_weighted_option_counts(
        question_text, question_id, structured_df, question2option
    )
    plot_weighted_hbar(
        option_counts, total_respondents, weighted_respondents, question_text
    )
    print(f"{question_id} counts:")
    print(option_counts)

    print(f"\n{question_id}:")
    print(f"  Weighted N = {weighted_respondents}")
    # Plot bar


### Statistical Tests

In [ ]:
import scipy.stats as stats
from scipy.stats import chi2_contingency


def run_chi_square_test(
    category1_counts, category2_counts, category1_name, category2_name
):
    """
    Run chi-square test for 2x2 contingency table with given counts.
    Returns chi-square statistic, p-value, and formatted results.
    """
    # Handle zero counts
    total = category1_counts + category2_counts
    if total == 0:
        return {
            "category1_name": category1_name,
            "category2_name": category2_name,
            "category1_count": category1_counts,
            "category2_count": category2_counts,
            "category1_pct": 0,
            "category2_pct": 0,
            "pct_difference": 0,
            "chi2_statistic": 0,
            "p_value": 1.0,
            "significant": False,
            "note": "No responses in either category",
        }

    # Expected values under null hypothesis (equal proportions)
    expected_each = total / 2

    # Calculate chi-square statistic manually for 2x2 case
    chi2_stat = (
        (category1_counts - expected_each) ** 2
        + (category2_counts - expected_each) ** 2
    ) / expected_each

    # Calculate p-value
    p_value = 1 - stats.chi2.cdf(chi2_stat, 1)

    # Calculate percentages
    pct1 = (category1_counts / total) * 100
    pct2 = (category2_counts / total) * 100
    pct_diff = pct1 - pct2

    return {
        "category1_name": category1_name,
        "category2_name": category2_name,
        "category1_count": category1_counts,
        "category2_count": category2_counts,
        "category1_pct": pct1,
        "category2_pct": pct2,
        "pct_difference": pct_diff,
        "chi2_statistic": chi2_stat,
        "p_value": p_value,
        "significant": p_value < 0.05,
    }


def run_mcnemar_test(both_selected, only_current_affairs, only_work_education, neither):
    """
    Run McNemar's test for dependent samples (multiple-selection question).
    """
    # Create 2x2 contingency table for McNemar's test
    # Focus on discordant pairs (b and c)
    b = only_current_affairs  # selected current affairs but not work/education
    c = only_work_education  # selected work/education but not current affairs

    # McNemar's test statistic with continuity correction
    mcnemar_stat = (abs(b - c) - 0.5) ** 2 / (b + c) if (b + c) > 0 else 0

    # p-value (chi-square distribution with df=1)
    p_value = 1 - stats.chi2.cdf(mcnemar_stat, 1) if (b + c) > 0 else 1.0

    # Calculate percentages
    total_responses = (
        both_selected + only_current_affairs + only_work_education + neither
    )
    pct_current = (
        ((both_selected + only_current_affairs) / total_responses) * 100
        if total_responses > 0
        else 0
    )
    pct_work = (
        ((both_selected + only_work_education) / total_responses) * 100
        if total_responses > 0
        else 0
    )
    pct_diff = pct_current - pct_work

    return {
        "current_affairs_total": both_selected + only_current_affairs,
        "work_education_total": both_selected + only_work_education,
        "both_selected": both_selected,
        "only_current_affairs": only_current_affairs,
        "only_work_education": only_work_education,
        "neither": neither,
        "pct_current_affairs": pct_current,
        "pct_work_education": pct_work,
        "pct_difference": pct_diff,
        "mcnemar_statistic": mcnemar_stat,
        "p_value": p_value,
        "significant": p_value < 0.05,
    }


def print_chi_square_results(result, test_name):
    """Print formatted results for chi-square test."""
    print(f"\n{test_name}")
    print("=" * 60)
    print(
        f"{result['category1_name']}: {result['category1_count']} ({result['category1_pct']:.1f}%)"
    )
    print(
        f"{result['category2_name']}: {result['category2_count']} ({result['category2_pct']:.1f}%)"
    )
    print(f"Difference: {result['pct_difference']:+.1f} percentage points")
    print(f"χ²(1) = {result['chi2_statistic']:.3f}, p = {result['p_value']:.3f}")
    print(f"Significant at p < 0.05: {'Yes' if result['significant'] else 'No'}")
    if "note" in result:
        print(f"Note: {result['note']}")


def print_mcnemar_results(result, test_name):
    """Print formatted results for McNemar's test."""
    print(f"\n{test_name}")
    print("=" * 60)
    print("Cross-tabulation breakdown:")
    print(f"  Both selected: {result['both_selected']}")
    print(f"  Only current affairs: {result['only_current_affairs']}")
    print(f"  Only work/education: {result['only_work_education']}")
    print(f"  Neither: {result['neither']}")
    print(
        f"\nCurrent affairs total: {result['current_affairs_total']} ({result['pct_current_affairs']:.1f}%)"
    )
    print(
        f"Work/education total: {result['work_education_total']} ({result['pct_work_education']:.1f}%)"
    )
    print(f"Difference: {result['pct_difference']:+.1f} percentage points")
    print(
        f"McNemar χ²(1) = {result['mcnemar_statistic']:.3f}, p = {result['p_value']:.3f}"
    )
    print(f"Significant at p < 0.05: {'Yes' if result['significant'] else 'No'}")


# Run statistical tests on survey data
print("STATISTICAL TESTS ON WEIGHTED SURVEY DATA")
print("=" * 80)

# First, get the data for all questions
test_questions = {
    "Q5b": "Q5b How useful, if at all, were the AI chatbot's replies?",
    "Q5c": "Q5c How accurate, if at all, did the AI chatbot's replies seem?",
    "Q5e": "Q5e Did the AI chatbot's replies seem to be fair and balanced, or did the replies favour left-wing views over right-wing views, or vice versa?",
    "Q5f": "Q5f Did the way the AI chatbot replied influence your perspective on the issues that you researched?",
    "Q3d": "Q3d Thinking about the information about UK current affairs or political issues that was generated by an AI chatbot, did it make you more likely to vote, less likely to vote, or did it make no difference?",
}

# Get weighted counts for each question
question_data = {}
for question_id, question_text in test_questions.items():
    option_counts, total_respondents, weighted_respondents = get_weighted_option_counts(
        question_text, question_id, structured_df, question2option
    )
    question_data[question_id] = {
        "counts": option_counts,
        "total": total_respondents,
        "weighted_total": weighted_respondents,
    }

# Debug: Print the actual option texts to see exact matches needed
print("\nDEBUG: Checking exact option texts for each question:")
for q_id, q_data in question_data.items():
    print(f"\n{q_id} options:")
    for option, count in q_data["counts"].items():
        print(f"  '{option}': {count}")

# Test 1 - Q5b: Useful vs Non-useful
print("\n\nTEST 1 - Q5b: USEFUL VS NON-USEFUL")
q5b_counts = question_data["Q5b"]["counts"]
useful = q5b_counts.get("Fairly useful", 0) + q5b_counts.get("Very useful", 0)
non_useful = q5b_counts.get("Not very useful", 0) + q5b_counts.get(
    "Not at all useful", 0
)
result1 = run_chi_square_test(useful, non_useful, "Useful", "Non-useful")
print_chi_square_results(result1, "Test 1 - Q5b: Useful vs Non-useful")

# Test 2 - Q5c: Accurate vs Non-accurate
print("\n\nTEST 2 - Q5c: ACCURATE VS NON-ACCURATE")
q5c_counts = question_data["Q5c"]["counts"]
accurate = q5c_counts.get("Fairly accurate", 0) + q5c_counts.get("Very accurate", 0)
non_accurate = q5c_counts.get("Not very accurate", 0) + q5c_counts.get(
    "Not at all accurate", 0
)
result2 = run_chi_square_test(accurate, non_accurate, "Accurate", "Non-accurate")
print_chi_square_results(result2, "Test 2 - Q5c: Accurate vs Non-accurate")

# Test 3 - Q5e: Politically balanced vs Partisan (aggregate)
print("\n\nTEST 3 - Q5e: BALANCED VS PARTISAN")
q5e_counts = question_data["Q5e"]["counts"]
balanced = q5e_counts.get(
    "The chatbot seemed to be politically neutral or balanced (it gave each side of the argument a fair hearing)",
    0,
)
partisan = q5e_counts.get(
    "The chatbot seemed to be politically right leaning", 0
) + q5e_counts.get("The chatbot seemed to be politically left leaning", 0)
result3 = run_chi_square_test(balanced, partisan, "Balanced", "Partisan")
print_chi_square_results(result3, "Test 3 - Q5e: Politically balanced vs Partisan")

# Test 4 - Q5e: Right vs Left leaning (subgroups)
print("\n\nTEST 4 - Q5e: RIGHT VS LEFT LEANING")
right_leaning = q5e_counts.get("The chatbot seemed to be politically right leaning", 0)
left_leaning = q5e_counts.get("The chatbot seemed to be politically left leaning", 0)
result4 = run_chi_square_test(
    right_leaning, left_leaning, "Right leaning", "Left leaning"
)
print_chi_square_results(result4, "Test 4 - Q5e: Right vs Left leaning")

# Test 5 - Q5f: Not influenced vs Influenced (aggregate)
print("\n\nTEST 5 - Q5f: NOT INFLUENCED VS INFLUENCED")
q5f_counts = question_data["Q5f"]["counts"]
not_influenced = q5f_counts.get("I was not influenced by the chatbot", 0)
# Check for exact text strings with periods
influenced = (
    q5f_counts.get("Yes, I was influenced in a more liberal direction.", 0)
    + q5f_counts.get("Yes, I was influenced in a more liberal direction", 0)
    + q5f_counts.get(
        "I was influenced by the chatbot, but not in a more liberal or conservative direction",
        0,
    )
    + q5f_counts.get("Yes, I was influenced in a more conservative direction.", 0)
    + q5f_counts.get("Yes, I was influenced in a more conservative direction", 0)
)
result5 = run_chi_square_test(
    not_influenced, influenced, "Not influenced", "Influenced"
)
print_chi_square_results(result5, "Test 5 - Q5f: Not influenced vs Influenced")

# Test 6 - Q5f: Conservative vs Liberal influence (subgroups)
print("\n\nTEST 6 - Q5f: CONSERVATIVE VS LIBERAL INFLUENCE")
# Check for exact text strings with periods
conservative_influence = q5f_counts.get(
    "Yes, I was influenced in a more conservative direction.", 0
) + q5f_counts.get("Yes, I was influenced in a more conservative direction", 0)
liberal_influence = q5f_counts.get(
    "Yes, I was influenced in a more liberal direction.", 0
) + q5f_counts.get("Yes, I was influenced in a more liberal direction", 0)
result6 = run_chi_square_test(
    conservative_influence,
    liberal_influence,
    "Conservative influence",
    "Liberal influence",
)
print_chi_square_results(result6, "Test 6 - Q5f: Conservative vs Liberal influence")

# Test 7 - Q3d: No voting influence vs Voting influence (aggregate)
print("\n\nTEST 7 - Q3d: NO VOTING INFLUENCE VS VOTING INFLUENCE")
q3d_counts = question_data["Q3d"]["counts"]
no_voting_influence = q3d_counts.get("It made no difference", 0)
voting_influence = q3d_counts.get("It made me more likely to vote", 0) + q3d_counts.get(
    "It made me less likely to vote", 0
)
result7 = run_chi_square_test(
    no_voting_influence, voting_influence, "No voting influence", "Voting influence"
)
print_chi_square_results(
    result7, "Test 7 - Q3d: No voting influence vs Voting influence"
)

# Test 8 - Q3d: More likely vs Less likely to vote (subgroups)
print("\n\nTEST 8 - Q3d: MORE LIKELY VS LESS LIKELY TO VOTE")
more_likely = q3d_counts.get("It made me more likely to vote", 0)
less_likely = q3d_counts.get("It made me less likely to vote", 0)
result8 = run_chi_square_test(
    more_likely, less_likely, "More likely to vote", "Less likely to vote"
)
print_chi_square_results(result8, "Test 8 - Q3d: More likely vs Less likely to vote")

# Test 9 - Q1e: McNemar's test for Current affairs vs Work/education
print("\n\nTEST 9 - Q1e: CURRENT AFFAIRS VS WORK/EDUCATION (McNEMAR'S TEST)")
# Get the raw data for Q1e to calculate cross-tabulation
q1e_text = "Q1e In the last four weeks, have you asked an AI chatbot for information or advice on any of the following topics?"
q1e_id = "Q1e"

# Get relevant variables for Q1e
relevant_vars = [col for col in structured_df.columns if col.startswith(q1e_id)]
weight_col = "W8"
id_col = "ID"

# Create dataframe with ID, weights, and Q1e responses
q1e_df = structured_df[[id_col, weight_col] + relevant_vars].copy()

# Remove "_other" columns
remove_cols = [col for col in q1e_df.columns if "_other" in col]
if remove_cols:
    q1e_df = q1e_df.drop(columns=remove_cols)

# Rename columns using the mapping
rename_dict = {k: v for k, v in question2option.items() if k in q1e_df.columns}
q1e_df = q1e_df.rename(columns=rename_dict)

# Handle empty strings
q1e_df = q1e_df.replace(r"^\s*$", np.nan, regex=True)

# Convert response columns to numeric
response_cols = [col for col in q1e_df.columns if col not in [id_col, weight_col]]
for col in response_cols:
    q1e_df[col] = pd.to_numeric(q1e_df[col], errors="coerce")

# Focus on current affairs and work/education columns
current_affairs_col = "UK current affairs or political issues (including information about the UK general election)"
work_education_col = "Information to help me with my work or education"

if current_affairs_col in q1e_df.columns and work_education_col in q1e_df.columns:
    # Create cross-tabulation
    # Filter to valid responses only
    valid_mask = (
        q1e_df[current_affairs_col].notna()
        & q1e_df[work_education_col].notna()
        & q1e_df[weight_col].notna()
    )

    valid_df = q1e_df[valid_mask].copy()

    # Fill NaN with 0 for calculation
    valid_df[current_affairs_col] = valid_df[current_affairs_col].fillna(0)
    valid_df[work_education_col] = valid_df[work_education_col].fillna(0)

    # Create categories
    both_mask = (valid_df[current_affairs_col] == 1) & (
        valid_df[work_education_col] == 1
    )
    only_current_mask = (valid_df[current_affairs_col] == 1) & (
        valid_df[work_education_col] == 0
    )
    only_work_mask = (valid_df[current_affairs_col] == 0) & (
        valid_df[work_education_col] == 1
    )
    neither_mask = (valid_df[current_affairs_col] == 0) & (
        valid_df[work_education_col] == 0
    )

    # Calculate weighted counts
    both_selected = (
        round(valid_df.loc[both_mask, weight_col].sum()) if both_mask.any() else 0
    )
    only_current_affairs = (
        round(valid_df.loc[only_current_mask, weight_col].sum())
        if only_current_mask.any()
        else 0
    )
    only_work_education = (
        round(valid_df.loc[only_work_mask, weight_col].sum())
        if only_work_mask.any()
        else 0
    )
    neither = (
        round(valid_df.loc[neither_mask, weight_col].sum()) if neither_mask.any() else 0
    )

    # Run McNemar's test
    result9 = run_mcnemar_test(
        both_selected, only_current_affairs, only_work_education, neither
    )
    print_mcnemar_results(
        result9, "Test 9 - Q1e: Current affairs vs Work/education (McNemar's Test)"
    )
else:
    print("Could not find required columns for McNemar's test")
    print("Available columns:", list(q1e_df.columns))

print("\n" + "=" * 80)
print("STATISTICAL TESTS COMPLETED")
print("=" * 80)

# Load and Clean Experiment Data

In [ ]:
#private beliefs: summary_full_GLM_private_prompting_0_no_model, Ztable_private_combined
#trust: summary_full_GLM_trust_prompting_0, Ztable_trust_combined
#extremism: summary_full_GLM_extreme_prompting_0, Ztable_extreme_combined

In [ ]:
# load parameter estimates
summary_df = pd.read_csv(
    "../parameter_estimates/summary_full_GLM_misinfo_prompting_0_no_model.csv"
)
summary_df


# Load waic df
waic_df = pd.read_csv("../parameter_estimates/waic_comparison_misinfo_prompting_0.csv")
# WAIC CSVs are now saved with a named index ("name"); older files used "Unnamed: 0".
waic_df = waic_df.rename({"Unnamed: 0": "model", "name": "model"}, axis=1)

# Print current working directory
print("Current working directory:", os.getcwd())
df = pd.read_csv("../data/Ztable_misinfo_combined.csv")
print(
    df[
        ["model", "searchornot", "iscontrol", "researched", "ground_truth"]
    ].value_counts()
)
# Keep only main models (using your original filtering)
exclude_models = ["GPT4o_sycophancy_both", "GPT4o_persuasion"]
df = df[~df["model"].isin(exclude_models)]
print(df["searchornot"].unique())

# Flip response variable
df["response"] = 8 - df["response"]

In [ ]:
df["iscontrol"].value_counts()

In [ ]:
df["subject"].nunique()

# Composite Plot

In [ ]:
truth_colors = {True: "#710462", False: "#fa5d0f"}  # Purple for True, Orange for False
line_styles = {"yes": "-", "no": ":"}  # Solid for researched, dotted for control
line_width = {"yes": 2.5, "no": 2.0}  # Thicker for researched, thinner for control
alpha_values = {
    "yes": 1.0,
    "no": 0.7,
}  # Full opacity for researched, semi-transparent for control
darker_truth_colors = {
    True: "#670359",  # Darker purple
    False: "#682402",  # Darker orange
}


# Helper function to add difference brackets
def add_difference_bracket(ax, x_pos, y1, y2, color, side="right", text_offset=-0.07):
    # Calculate difference
    diff = round(y1 - y2, 2)
    diff_text = f"Δ = {diff:+.2f}"

    # Set bracket width based on side
    bracket_width = 0.1
    if side == "right":
        x_start = x_pos + 0.05
        text_x = x_start + bracket_width + text_offset
        ha = "left"
    else:  # left side
        x_start = x_pos - 0.05
        text_x = x_start - bracket_width - text_offset
        ha = "right"

    # Draw vertical lines
    ax.plot(
        [x_start, x_start],
        [y1, y2],
        color=color,
        linestyle="-",
        linewidth=1.5,
        alpha=0.7,
    )

    # Add horizontal end caps - facing outwards
    cap_length = 0.02
    if side == "right":
        ax.plot(
            [x_start, x_start - cap_length],
            [y1, y1],
            color=color,
            linewidth=2,
            alpha=0.7,
        )
        ax.plot(
            [x_start, x_start - cap_length],
            [y2, y2],
            color=color,
            linewidth=2,
            alpha=0.7,
        )
    else:
        ax.plot(
            [x_start, x_start + cap_length],
            [y1, y1],
            color=color,
            linewidth=2,
            alpha=0.7,
        )
        ax.plot(
            [x_start, x_start + cap_length],
            [y2, y2],
            color=color,
            linewidth=2,
            alpha=0.7,
        )

    # Add difference text
    y_text = (y1 + y2) / 2
    ax.text(
        text_x,
        y_text,
        diff_text,
        color=color,
        fontweight="bold",
        ha=ha,
        va="center",
        fontsize=20,
        bbox=dict(facecolor="white", alpha=0.7, edgecolor="none", pad=1),
    )


# Helper function to add delta-of-deltas annotation with horizontal bracket
def add_delta_of_deltas(ax, pre_delta, post_delta, color, y_pos, cap_direction="down"):
    # Calculate the difference between post delta and pre delta
    delta_of_deltas = round(post_delta - pre_delta, 2)

    # Create the annotation text
    delta_text = f"ΔΔ = {delta_of_deltas:+.2f}"

    # Position for the horizontal bracket
    x_left = 0.0  # Position at pre delta
    x_right = 1.0  # Position at post delta

    # Draw the horizontal line connecting the deltas
    ax.plot(
        [x_left, x_right],
        [y_pos, y_pos],
        color=color,
        linestyle="-",
        linewidth=1.5,
        alpha=0.7,
        zorder=5,
    )

    # Add vertical end caps to the bracket, direction based on parameter
    cap_length = 0.05
    if cap_direction == "down":
        # Caps pointing down
        ax.plot(
            [x_left, x_left],
            [y_pos, y_pos - cap_length],
            color=color,
            linewidth=1.5,
            alpha=0.7,
            zorder=5,
        )
        ax.plot(
            [x_right, x_right],
            [y_pos, y_pos - cap_length],
            color=color,
            linewidth=1.5,
            alpha=0.7,
            zorder=5,
        )
    else:
        # Caps pointing up
        ax.plot(
            [x_left, x_left],
            [y_pos, y_pos + cap_length],
            color=color,
            linewidth=1.5,
            alpha=0.7,
            zorder=5,
        )
        ax.plot(
            [x_right, x_right],
            [y_pos, y_pos + cap_length],
            color=color,
            linewidth=1.5,
            alpha=0.7,
            zorder=5,
        )

    # Add the delta-of-deltas text
    text_y_pos = y_pos + 0.01 if cap_direction == "down" else y_pos - 0.03
    text_va = "bottom" if cap_direction == "down" else "top"

    ax.text(
        0.5,  # Center between pre and post
        text_y_pos,
        delta_text,
        color=color,
        fontweight="bold",
        ha="center",
        va=text_va,
        fontsize=12,
        bbox=dict(facecolor="white", alpha=0.0, edgecolor="none", pad=2),
        zorder=10,  # Make sure it's on top
    )


# Dictionary to store values for comparing research vs control
comparison_values = {
    "test": {"pre": {}, "post": {}},  # Conversational AI
    "control": {"pre": {}, "post": {}},  # Search
}

# Dictionary to store delta values for delta-of-deltas calculation
delta_values = {
    "test": {"pre": {}, "post": {}},  # Conversational AI
    "control": {"pre": {}, "post": {}},  # Search
}

In [ ]:
def create_bar_chart(ax, data_df, total_users_new, title, big_fontsize, med_fontsize):
    """Helper function to create a bar chart with the first bar in red and others in dark grey"""
    # Calculate percentages
    percentages = (data_df / total_users_new * 100).round(1)

    df_plot = pd.DataFrame(
        {
            "option": data_df.index,
            "count": data_df.values,
            "percentage": percentages.values,
        }
    )

    # Sort the dataframe by count in descending order
    df_plot = df_plot.sort_values("count", ascending=False)

    # Create custom colors list - first bar red, rest dark grey
    colors = ["#B7003F"] + ["#708090"] * (len(df_plot) - 1)

    # Create the bar plot
    sns.barplot(
        data=df_plot, x="count", y="option", palette=colors, legend=False, ax=ax
    )

    # Add percentage labels on the bars
    for i, (count, percentage) in enumerate(
        zip(df_plot["count"], df_plot["percentage"])
    ):
        ax.text(
            count - 15,
            i,
            f"{percentage}%",
            va="center",
            ha="right",
            fontsize=med_fontsize,
            color="white",
            fontweight="semibold",
        )

    # Set title and formatting
    ax.set_title(title, fontsize=big_fontsize, fontweight="semibold")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xlim(0, max(data_df.values) * 1.1)
    ax.tick_params(axis="both", labelsize=med_fontsize)

In [ ]:
def create_composite_grid_plot(
    survey_usecases_df,
    total_users_usecases,
    question_text,
    df,
    comparison_values,
    delta_values,
    truth_colors,
    line_styles,
    line_width,
    alpha_values,
    darker_truth_colors,
    summary_df,
    waic_df,
    rope=0.05,
    figsize=(20, 15),
    big_fontsize=16,
    med_fontsize=14,
    small_fontsize=12,
):
    """
    Create a composite grid plot with bar chart, line plots, and model parameters

    Layout:
    - Left third (full height): Bar chart
    - Right upper two-thirds: Line plots (Conversational AI and Search)
    - Right bottom third: Model parameter estimates and WAIC comparison
    """
    # Create figure with custom grid layout
    fig = plt.figure(figsize=figsize)

    gs = fig.add_gridspec(
        3,
        4,
        width_ratios=[1, 0.05, 1.1, 1.1],
        height_ratios=[2, 1.5, 1.5],
        wspace=0.8,
        hspace=0.8,
    )

    # Bar chart (first column)
    # Bar charts (first column - split into two)
    ax_bar_top = fig.add_subplot(gs[:, 0])  # Top 1/3

    # Column 1 is empty (no subplot created for it) - this creates the white space
    # Line plots (right two columns)
    ax_conv = fig.add_subplot(gs[0:2, 2])
    ax_search = fig.add_subplot(gs[0:2, 3])

    # Model parameters (right two columns)
    ax_params = fig.add_subplot(gs[2, 2])
    ax_waic = fig.add_subplot(gs[2, 3])

    # === HORIZONTAL BAR CHART ===
    # Get first color from Reds palette
    color_1 = sns.color_palette("Reds_r", n_colors=1)[0]  # Get first
    color_0 = sns.color_palette("Blues_r", n_colors=1)[0]  # Get first
    palette = [color_0, color_1]
    # create_bar_chart(ax_bar_top, survey_use_df, total_users_use, palette=palette, title = f"Sources of Political Information\nN={total_users_use}")

    # Bottom bar chart (Use cases)
    create_bar_chart(
        ax_bar_top,
        survey_usecases_df,
        total_users_usecases,
        title=f"Usecases of AI Chatbots\n(N = 1,024)",
        big_fontsize=big_fontsize,
        med_fontsize=med_fontsize,
    )

    # # Set annotation for the top bar chart
    # ax_bar_top.annotate("UK Pre-Election Survey Results", fontsize=16, fontweight="bold",
    #                     xy=(0.5, 1.2), xycoords='axes fraction', ha='center', va='bottom')

    for ax in [ax_bar_top]:
        ax.spines["right"].set_visible(False)
        ax.spines["top"].set_visible(False)
        ax.spines["bottom"].set_visible(False)

    # === LINE PLOTS ===
    # Set up axes for line plots
    axes = [ax_conv, ax_search]

    # Process data and store values for later comparison
    for condition in df["iscontrol"].unique():
        df_condition = df[df["iscontrol"] == condition]
        N = df_condition["subject"].nunique()

        for researched in df_condition["researched"].unique():
            df_researched = df_condition[df_condition["researched"] == researched]

            # Select the appropriate subplot
            if condition == "control":  # Search
                ax = ax_search
                title = f"Search\n(N = {N})"
            elif condition == "test":  # Conversational AI
                ax = ax_conv
                title = f"Conversational AI\n(N = {N})"

            # Set improved title
            ax.set_title(title, fontsize=big_fontsize, fontweight="bold", pad=10)

            # Turn off all x gridlines and keep dotted faint y gridlines
            ax.xaxis.grid(False)
            ax.yaxis.grid(True, linestyle="--", alpha=0.7)

            # Add X-axis padding by setting limits explicitly
            ax.set_xlim(-0.35, 1.35)  # Increase padding for the brackets
            ax.set_ylim(3.6, 4.8)

            # Plot separate lines for each truth value
            for groundtruth in df_researched["ground_truth"].unique():
                df_groundtruth = df_researched[
                    df_researched["ground_truth"] == groundtruth
                ]

                pre_responses = df_groundtruth[df_groundtruth["presented"] == "pre"]
                post_responses = df_groundtruth[df_groundtruth["presented"] == "post"]

                # Skip if empty
                if pre_responses.empty or post_responses.empty:
                    continue

                # Get the color based on the truth value
                color = truth_colors[bool(groundtruth)]

                # Calculate means
                mean_pre = pre_responses["response"].mean()
                mean_post = post_responses["response"].mean()

                # Store values for later comparison between researched and control
                comparison_values[condition]["pre"].setdefault(bool(groundtruth), {})
                comparison_values[condition]["post"].setdefault(bool(groundtruth), {})
                comparison_values[condition]["pre"][bool(groundtruth)][
                    researched
                ] = mean_pre
                comparison_values[condition]["post"][bool(groundtruth)][
                    researched
                ] = mean_post

                # Plot line between pre and post with enhanced styling based on researched status
                line = ax.plot(
                    ["Pre", "Post"],
                    [mean_pre, mean_post],
                    marker="o",
                    markersize=10,
                    markeredgecolor="white",
                    markeredgewidth=1.5,
                    color=color,
                    linestyle=line_styles[researched],
                    linewidth=line_width[researched],
                    alpha=alpha_values[researched],
                )[0]

                # Calculate confidence intervals
                pre_std = pre_responses["response"].std()
                post_std = post_responses["response"].std()
                pre_n = pre_responses["response"].count()
                post_n = post_responses["response"].count()
                pre_error = 1.96 * (pre_std / np.sqrt(pre_n))
                post_error = 1.96 * (post_std / np.sqrt(post_n))

                # Add error bars with improved styling
                ax.errorbar(
                    "Pre",
                    mean_pre,
                    yerr=pre_error,
                    fmt="none",
                    ecolor=color,
                    capsize=5,
                    elinewidth=1.5,
                    alpha=alpha_values[researched],
                )
                ax.errorbar(
                    "Post",
                    mean_post,
                    yerr=post_error,
                    fmt="none",
                    ecolor=color,
                    capsize=5,
                    elinewidth=1.5,
                    alpha=alpha_values[researched],
                )

            # Format the axes
            ax.set_xticks(["Pre", "Post"])
            ax.set_xticklabels(
                ["Pre", "Post"], fontsize=med_fontsize, fontweight="bold"
            )
            ax.xaxis.grid(False)
            ax.yaxis.grid(True, linestyle="--", alpha=0.8)

            # Add y-axis label only to the leftmost plot
            if ax == ax_conv:
                ax.set_ylabel(
                    "Mean Likert\n(1=disagree,\n7=agree)",
                    fontsize=med_fontsize,
                    labelpad=70,
                    rotation=0,
                    ha="center",
                )

            # Remove top and right spines
            ax.spines["right"].set_visible(False)
            ax.spines["top"].set_visible(False)
            ax.spines["bottom"].set_visible(False)

            # Style bottom and left spines
            ax.spines["bottom"].set_color("gray")
            ax.spines["left"].set_color("gray")
            ax.spines["bottom"].set_linewidth(0.5)
            ax.spines["left"].set_linewidth(0.5)
            ax.tick_params(axis="y", labelsize=med_fontsize)

    # Add difference brackets for post values only (right side)
    for condition in ["test", "control"]:
        if condition == "test":
            ax = ax_conv  # Conversational AI panel
        else:
            ax = ax_search  # Search panel

        # Add brackets for post only
        for truth_value in [True, False]:
            if (
                "yes" in comparison_values[condition]["post"][truth_value]
                and "no" in comparison_values[condition]["post"][truth_value]
            ):
                researched_val = comparison_values[condition]["post"][truth_value][
                    "yes"
                ]
                control_val = comparison_values[condition]["post"][truth_value]["no"]
                add_difference_bracket(
                    ax,
                    1,  # position at post
                    researched_val,
                    control_val,
                    darker_truth_colors[truth_value],
                    side="right",
                )

        for ax in [ax_conv, ax_search]:
            # Add thicker mid point line at y=4
            ax.axhline(4, color="black", linestyle="-", linewidth=1.5, alpha=0.7)

    # Legend

    truth_elements = [
        Line2D([0], [0], color=truth_colors[True], lw=6, label="True Information"),
        Line2D([0], [0], color=truth_colors[False], lw=6, label="False Information"),
    ]

    control_elements = [
        Line2D(
            [0], [0], color="black", lw=3, linestyle="-", label="Treatment (Researched)"
        ),
        Line2D(
            [0],
            [0],
            color="black",
            lw=3,
            linestyle=":",
            label="Control (Not Researched)",
        ),
    ]

    # Add legends below the line plots
    legend1 = fig.legend(
        handles=truth_elements,
        loc="upper center",
        bbox_to_anchor=(0.7, 1.06),
        ncol=2,
        frameon=False,
        fontsize=25,
    )
    legend2 = fig.legend(
        handles=control_elements,
        loc="upper center",
        bbox_to_anchor=(0.7, 1.03),
        ncol=2,
        frameon=False,
        fontsize=25,
    )

    # === PARAMETER ESTIMATES ===
    # Define ROPE interval
    rope_interval = np.array([-rope, rope])

    # Filter for specific parameters of interest
    params_of_interest = [
        "ground_truth",
        "time_ground_truth_researched",
        "time_ground_truth_researched_istest",
    ]

    # Filter the dataframe
    plot_df = summary_df[summary_df["Parameter"].isin(params_of_interest)].copy()

    # Rename parameters for better clarity
    param_names = {
        "ground_truth": "True info",
        "time_ground_truth_researched": "Post × True × Researched",
        "time_ground_truth_researched_istest": "Post × True × Researched × ConvAI",
    }

    plot_df["Parameter"] = plot_df["Parameter"].map(lambda x: param_names.get(x, x))

    # Calculate xerr as the difference from the mean to the confidence bounds
    xerr = np.abs(plot_df[["HDI_2.5%", "HDI_97.5%"]].T.values - plot_df["Mean"].values)

    # Reverse the order (True vs Misinfo at top)
    plot_df = plot_df.iloc[::-1].reset_index(drop=True)
    xerr = xerr[:, ::-1]

    # Create forest plot
    ax_params.errorbar(
        plot_df["Mean"],
        range(plot_df.shape[0]),
        xerr=xerr,
        fmt="o",
        color="black",
        markersize=8,
        capsize=5,
    )

    # Add a vertical line at 0
    ax_params.axvline(0, color="gray", linestyle="--", linewidth=0.5)

    # Add the shaded zone for the ROPE
    rope_rect = ax_params.axvspan(
        rope_interval[0], rope_interval[1], color="gray", alpha=0.3, label="ROPE ±0.05"
    )

    # Add labels with vertical alignment
    ax_params.set_yticks(range(plot_df.shape[0]))
    ax_params.set_yticklabels(plot_df["Parameter"], ha="right", fontsize=med_fontsize)

    # Add effect size values at the end of each bar
    for i, (_, row) in enumerate(plot_df.iterrows()):
        ax_params.text(
            row["Mean"] + xerr[1, i] + 0.02,  # Position after the error bar
            i,
            f"{row['Mean']:.3f}",
            va="center",
            ha="left",
            fontsize=med_fontsize,
        )

    # Add labels and title
    ax_params.set_xlabel("Effect Size", fontsize=med_fontsize)
    ax_params.set_title("Parameter Estimates", fontsize=big_fontsize, fontweight="bold")

    # Add proper legend for ROPE     # set col spacing

    ax_params.legend(
        handles=[rope_rect],
        loc="upper center",
        bbox_to_anchor=(0.5, 1.5),  # Position above the subplot
        fontsize=med_fontsize,
        handletextpad=0.2,
        frameon=False,
    )

    # Add grid lines for better readability
    ax_params.grid(False)  # , axis='x', linestyle='--', alpha=0.5)

    # Set optimized x-axis limits from -0.2 to 0.8
    ax_params.set_xlim(-0.2, 0.8)
    ax_params.margins(0.2)

    # Remove top and right spines
    ax_params.spines["right"].set_visible(False)
    ax_params.spines["top"].set_visible(False)
    ax_params.tick_params(axis="both", labelsize=med_fontsize)

    # === WAIC COMPARISON ===
    # Get model names from the index
    models = waic_df["model"].tolist()

    # Create better model names for display
    model_display_names = {
        "GLM_full_with_model": "Full Model",
        "GLM_full_without_model": "No ConvAI Term",
        "GLM_control": "Null Model",
    }

    # Map to nicer display names
    display_names = [model_display_names.get(model, model) for model in models]

    # Create y-positions for the models (0, 1, 2)
    y_pos = range(len(models))

    # Extract ELPD values and standard errors
    elpd_values = waic_df["elpd_waic"].values
    se_values = waic_df["se"].values

    # Plot the points with white fill, black edge
    for i, (elpd, se, name) in enumerate(zip(elpd_values, se_values, display_names)):
        # Plot error bars
        ax_waic.plot([elpd - se, elpd + se], [i, i], "-", color="black", linewidth=1.5)

        # Add vertical caps to the error bars
        cap_length = 0.05
        ax_waic.plot(
            [elpd - se, elpd - se],
            [i - cap_length, i + cap_length],
            "-",
            color="black",
            linewidth=1.5,
        )
        ax_waic.plot(
            [elpd + se, elpd + se],
            [i - cap_length, i + cap_length],
            "-",
            color="black",
            linewidth=1.5,
        )

        # Plot white-filled marker with black edge
        ax_waic.plot(
            elpd,
            i,
            "o",
            color="black",
            markerfacecolor="white",
            markersize=8,
            markeredgewidth=1.5,
        )

    # Add model names on the y-axis
    ax_waic.set_yticks(y_pos)
    ax_waic.set_yticklabels(display_names, fontsize=med_fontsize)

    # Add title and improved label
    ax_waic.set_title("Model Fit Comparison", fontsize=big_fontsize, fontweight="bold")
    ax_waic.set_xlabel("WAIC", fontsize=med_fontsize)

    # Optimize x-axis range
    best_elpd = max(elpd_values)
    min_elpd = min(elpd_values)
    elpd_range = best_elpd - min_elpd

    # Set the range to include just what's needed (with minimal padding)
    min_with_error = min(elpd_values - se_values)
    max_with_error = max(elpd_values + se_values)

    # Round to nearest 100 for cleaner ticks
    x_min = np.floor(min_with_error / 100) * 100
    x_max = np.ceil(max_with_error / 100) * 100

    # Create ticks at sensible intervals
    tick_interval = 5  # Determine a good interval based on the range
    if elpd_range > 50:
        tick_interval = 10

    # Generate ticks
    ax_waic.xaxis.set_major_locator(plt.MaxNLocator(4))  # Maximum 4 ticks
    ax_waic.grid(False, axis="x", linestyle="--", alpha=0.5, which="major")
    ax_waic.grid(False, which="minor")  # Turn off minor gridlines

    # Set x limits
    padding = elpd_range * 0.05  # Small padding
    ax_waic.set_xlim(x_min - padding, x_max + padding)
    ax_waic.tick_params(axis="both", labelsize=med_fontsize)

    # Add vertical reference line at the best model's ELPD
    ax_waic.axvline(best_elpd, color="gray", linestyle="--", alpha=0.75)

    # Add "closer to 0 is better" annotation
    ax_waic.text(
        0.98,
        0.3,
        "closer to 0 is better",
        ha="right",
        va="top",
        fontsize=med_fontsize,
        transform=ax_waic.transAxes,
        bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"),
    )

    # Add grid for better readability
    ax_waic.grid(False)  # , axis='x', linestyle='--', alpha=0.5)

    # Remove top and right spines
    ax_waic.spines["right"].set_visible(False)
    ax_waic.spines["top"].set_visible(False)

    # Add panel labels A, B, C with circles
    # Panel A (Bar chart)
    ax_bar_top.annotate(
        "A",
        xy=(-0.4, 1.07),
        xycoords="axes fraction",
        fontsize=35,
        fontweight="bold",
        bbox=dict(boxstyle="circle", fc="white", ec="black", alpha=0.8),
    )

    # Panel B (Line plots - Conversational AI)
    ax_conv.annotate(
        "B",
        xy=(-0.3, 1.11),
        xycoords="axes fraction",
        fontsize=35,
        fontweight="bold",
        bbox=dict(boxstyle="circle", fc="white", ec="black", alpha=0.8),
    )

    # Panel C (Model parameters)
    ax_params.annotate(
        "C",
        xy=(-0.3, 1.25),
        xycoords="axes fraction",
        fontsize=35,
        fontweight="bold",
        bbox=dict(boxstyle="circle", fc="white", ec="black", alpha=0.8),
    )

    # Fine-tune the layout
    plt.tight_layout()
    plt.subplots_adjust(hspace=0.3, wspace=0.5)

    return fig

In [ ]:
option2label = {
    "UK current affairs or political issues (including information about the UK general election)": "UK current affairs\nor political issues",
    "Information to help me with my work or education": "Work or education",
    "Help with translation, or help composing a piece of writing": "Writing\nor translation",
    "I used a chatbot for something else": "Other uses",
    "Practical matters, such as DIY or cooking": "Practical matters\n(DIY, cooking)",
    "I tried to engage the chatbot in a conversation just for fun": "Just for fun",
    "Personal issues, such as health or relationships": "Personal issues\n(health, relationships)",
    "Legal or financial advice": "Legal\nor financial advice",
    "Internet Search (for example Google)": "Internet Search\n(e.g., Google)",
    "AI chatbots (for example ChatGPT)": "AI chatbots\n(e.g., ChatGPT)",
}

survey_usecases_rn = survey_usecases.rename(index=option2label)

In [ ]:
# Use a serif font
plt.rcParams["font.family"] = "serif"
fig = create_composite_grid_plot(
    survey_usecases_rn,
    total_users_usecases,
    question_text,
    df,
    comparison_values,
    delta_values,
    truth_colors,
    line_styles,
    line_width,
    alpha_values,
    darker_truth_colors,
    summary_df,
    waic_df,
    rope=0.05,
    figsize=(27, 15),
    big_fontsize=26,
    med_fontsize=22,
    small_fontsize=18,
)

# Save as pdf
fig.savefig(
    "../figures/main-fig.pdf",
    bbox_inches="tight",
    format="pdf",
    dpi=300,
)

# Plot other outcomes (Private beliefs, trust, extremity)

In [ ]:
def create_line_and_params_plot(
    df_dict,  # Dictionary with 'private': df, 'trust': df, 'extremism': df
    summary_dict,  # Dictionary with 'private': summary_df, 'trust': summary_df, 'extremism': summary_df
    ztable_dict,  # Dictionary with 'private': ztable_df, 'trust': ztable_df, 'extremism': ztable_df
    variable_names,  # Dictionary with display names for each variable
    truth_colors,
    line_styles,
    line_width,
    alpha_values,
    darker_truth_colors,
    rope=0.05,
    figsize=(20, 12),
    big_fontsize=16,
    med_fontsize=14,
    small_fontsize=12,
):
    """
    Create a grid plot with line plots and parameter estimates
    
    Layout:
    - Top row: Trust line plots (ConvAI + Search), Private beliefs line plots (ConvAI + Search)
    - Bottom row: Trust parameter estimates, Private beliefs + Extremism parameter estimates
    """
    
    # Create figure with custom grid layout
    fig = plt.figure(figsize=figsize)
    
    # Set font properties to match the reference
    plt.rcParams.update({
        'font.family': 'serif',
        'font.serif': ['Times New Roman', 'Times', 'DejaVu Serif'],
        'font.size': med_fontsize,
        'axes.labelsize': med_fontsize,
        'axes.titlesize': big_fontsize,
        'xtick.labelsize': small_fontsize,
        'ytick.labelsize': small_fontsize,
        'legend.fontsize': med_fontsize,
    })

    # 2 rows, 5 columns (with gap column)
    gs = fig.add_gridspec(
        2, 5,
        width_ratios=[1, 1, 0.6, 1, 1],  # Trust plots, gap, Private plots
        height_ratios=[2, 1.5],  # Line plots taller than parameter plots
        wspace=0.4,  # Small spacing everywhere else
        hspace=0.4,
    )
    
    # Only process trust and private for line plots
    variables = ['trust', 'private']
    comparison_values = {}
    
    # Create subplots for trust and private beliefs line plots
    for i, var in enumerate(variables):
        # Line plots (top row)
        # ax_conv = fig.add_subplot(gs[0, i*2])      # Conversational AI
        # ax_search = fig.add_subplot(gs[0, i*2+1])  # Search
        if i == 0:  # Trust
            ax_conv = fig.add_subplot(gs[0, 0])      # Conversational AI
            ax_search = fig.add_subplot(gs[0, 1])    # Search
        else:  # Private (i == 1)
            ax_conv = fig.add_subplot(gs[0, 3])      # Conversational AI
            ax_search = fig.add_subplot(gs[0, 4])    # Search

        
        # Get data for this variable
        df = ztable_dict[var]
        
        # Check if we have data for this variable
        if df.empty:
            # If no data, add a text message to the plots
            for ax in [ax_conv, ax_search]:
                ax.text(0.5, 0.5, f'No data available\nfor {variable_names[var]}', 
                       ha='center', va='center', transform=ax.transAxes,
                       fontsize=med_fontsize)
                ax.set_xlim(0, 1)
                ax.set_ylim(0, 1)
                # Remove ticks and labels
                ax.set_xticks([])
                ax.set_yticks([])
                # Style spines
                for spine in ax.spines.values():
                    spine.set_visible(False)
            continue
        
        # Initialize comparison values for this variable
        comparison_values[var] = {"test": {"pre": {}, "post": {}}, "control": {"pre": {}, "post": {}}}
        
        # === LINE PLOTS ===
        # First, set titles for the subplots
        ax_conv.set_title("Conversational AI", fontsize=big_fontsize, fontweight="bold", pad=10)
        ax_search.set_title("Search", fontsize=big_fontsize, fontweight="bold", pad=10)
        
        # Track if we actually plot any data
        data_plotted = False
        
        # Process data for line plots - Handle both string and numeric iscontrol values
        for condition in df["iscontrol"].unique():
            df_condition = df[df["iscontrol"] == condition]
            N = df_condition["subject"].nunique()
            
            print(f"Processing condition: {condition} (type: {type(condition)})")
            
            # Select the appropriate subplot based on condition - Handle both formats
            ax = None
            condition_name = None
            
            # Handle different encodings of iscontrol
            if condition == 0 or condition == "control":  # Search/control
                ax = ax_search
                condition_name = "control"
            elif condition == 1 or condition == "test":  # Conversational AI/test
                ax = ax_conv
                condition_name = "test"
            else:
                print(f"Unknown condition: {condition}")
                continue  # Skip unknown conditions
            
            print(f"Mapped to condition_name: {condition_name}")
            
            for researched in df_condition["researched"].unique():
                df_researched = df_condition[df_condition["researched"] == researched]
                
                # Plot lines for each truth value
                for groundtruth in df_researched["ground_truth"].unique():
                    df_groundtruth = df_researched[df_researched["ground_truth"] == groundtruth]
                    
                    pre_responses = df_groundtruth[df_groundtruth["presented"] == "pre"]
                    post_responses = df_groundtruth[df_groundtruth["presented"] == "post"]
                    
                    print(f"  Plotting {var}-{condition_name}-{researched}-{groundtruth}: pre={len(pre_responses)}, post={len(post_responses)}")
                    
                    if pre_responses.empty or post_responses.empty:
                        print(f"    Skipping - empty data")
                        continue
                    
                    # For trust: ground_truth True = Distrust statements, False = Trust statements
                    if var == 'trust':
                        # Use different colors for trust vs distrust statements
                        if bool(groundtruth):  # True = Distrust statements
                            color = truth_colors[True] 
                            statement_type = "Distrust"
                        else:  # False = Trust statements  
                            color = truth_colors[False]
                            statement_type = "Trust"
                    else:
                        # For other variables, use normal true/false coloring
                        color = truth_colors[bool(groundtruth)]
                    
                    # Calculate means
                    mean_pre = pre_responses["response"].mean()
                    mean_post = post_responses["response"].mean()
                    
                    print(f"    Means: pre={mean_pre:.2f}, post={mean_post:.2f}")
                    
                    # Store values for comparison
                    comparison_values[var][condition_name]["pre"].setdefault(bool(groundtruth), {})
                    comparison_values[var][condition_name]["post"].setdefault(bool(groundtruth), {})
                    comparison_values[var][condition_name]["pre"][bool(groundtruth)][researched] = mean_pre
                    comparison_values[var][condition_name]["post"][bool(groundtruth)][researched] = mean_post
                    
                    # Plot line using numeric positions (0 for Pre, 1 for Post)
                    line = ax.plot(
                        [0, 1],  # Use numeric positions instead of strings
                        [mean_pre, mean_post],
                        marker="o",
                        markersize=10,
                        markeredgecolor="white",
                        markeredgewidth=1.5,
                        color=color,
                        linestyle=line_styles[researched],
                        linewidth=line_width[researched],
                        alpha=alpha_values[researched],
                    )
                    
                    print(f"    Plotted line successfully")
                    
                    # Calculate and add error bars
                    pre_std = pre_responses["response"].std()
                    post_std = post_responses["response"].std()
                    pre_n = pre_responses["response"].count()
                    post_n = post_responses["response"].count()
                    pre_error = 1.96 * (pre_std / np.sqrt(pre_n))
                    post_error = 1.96 * (post_std / np.sqrt(post_n))
                    
                    ax.errorbar(0, mean_pre, yerr=pre_error, fmt="none", ecolor=color, 
                               capsize=5, elinewidth=1.5, alpha=alpha_values[researched])
                    ax.errorbar(1, mean_post, yerr=post_error, fmt="none", ecolor=color,
                               capsize=5, elinewidth=1.5, alpha=alpha_values[researched])
                    
                    data_plotted = True
        
        print(f"Data plotted for {var}: {data_plotted}")
        
        # Format both line plot axes
        for ax in [ax_conv, ax_search]:
            # Grid and axis settings
            ax.xaxis.grid(False)
            ax.yaxis.grid(True, linestyle="--", alpha=0.7)
            ax.set_xlim(-0.35, 1.35)
            ax.set_ylim(3.4, 4.8)
            
            # Format axes using numeric positions
            ax.set_xticks([0, 1])  # Use numeric positions
            ax.set_xticklabels(["Pre", "Post"], fontsize=med_fontsize, fontweight="bold")
            
            # Style spines
            ax.spines["right"].set_visible(False)
            ax.spines["top"].set_visible(False)
            ax.spines["bottom"].set_visible(False)
            ax.spines["bottom"].set_color("gray")
            ax.spines["left"].set_color("gray")
            ax.spines["bottom"].set_linewidth(0.5)
            ax.spines["left"].set_linewidth(0.5)
            ax.tick_params(axis="y", labelsize=med_fontsize)
            
            # Add midpoint line
            ax.axhline(4, color="black", linestyle="-", linewidth=1.5, alpha=0.7)
        
        # Add y-axis label only to the leftmost plot
        if i == 0:
            # Different y-label for trust vs other variables
            if var == 'trust':
                ylabel = "Mean Agreement\n(1=disagree,\n7=agree)"
            else:
                ylabel = "Mean Likert\n(1=disagree,\n7=agree)"
                
            ax_conv.set_ylabel(
                ylabel,
                fontsize=med_fontsize,
                labelpad=70,
                rotation=0,
                ha="center",
            )
        
        # Add variable name above each pair of line plots
        if i == 0:  # Trust
            fig.text(0.27, 0.93, variable_names[var], fontsize=big_fontsize+2, 
                    fontweight="bold", ha="center", transform=fig.transFigure)
            # Add panel A label to Trust section
            ax_conv.annotate(
                "A",
                xy=(-0.3, 1.07),
                xycoords="axes fraction",
                fontsize=27,
                fontweight="bold",
                bbox=dict(boxstyle="circle", fc="white", ec="black", alpha=0.8),
            )
        else:  # Private Beliefs
            fig.text(0.75, 0.93, variable_names[var], fontsize=big_fontsize+2, 
                    fontweight="bold", ha="center", transform=fig.transFigure)
            # Add panel B label to Private Beliefs section
            ax_conv.annotate(
                "B",
                xy=(-0.3, 1.07),
                xycoords="axes fraction",
                fontsize=27,
                fontweight="bold",
                bbox=dict(boxstyle="circle", fc="white", ec="black", alpha=0.8),
            )
    
    # === PARAMETER ESTIMATES ===
    # Bottom row: Trust parameters (left) and Private + Extremism parameters (right)
    ax_trust_params = fig.add_subplot(gs[1, 0:2])  # Span first 2 columns
    ax_combined_params = fig.add_subplot(gs[1, 3:5])  # Span last 2 columns


    # Trust parameters
    var = 'trust'
    summary_df = summary_dict[var]
    
    params_of_interest = [
        "ground_truth",
        "time_ground_truth_researched", 
        "time_ground_truth_researched_istest",
    ]
    
    plot_df = summary_df[summary_df["Parameter"].isin(params_of_interest)].copy()
    
    if not plot_df.empty:
        param_names = {
            "ground_truth": "Distrust",
            "time_ground_truth_researched": "Post × Distrust × Researched",
            "time_ground_truth_researched_istest": "Post × Distrust × Researched × ConvAI",
        }
        
        plot_df["Parameter"] = plot_df["Parameter"].map(lambda x: param_names.get(x, x))
        xerr = np.abs(plot_df[["HDI_2.5%", "HDI_97.5%"]].T.values - plot_df["Mean"].values)
        plot_df = plot_df.iloc[::-1].reset_index(drop=True)
        xerr = xerr[:, ::-1]
        
        ax_trust_params.errorbar(
            plot_df["Mean"],
            range(plot_df.shape[0]),
            xerr=xerr,
            fmt="o",
            color="black",
            markersize=8,
            capsize=5,
        )
        
        ax_trust_params.axvline(0, color="gray", linestyle="--", linewidth=0.5)
        ax_trust_params.axvspan(-rope, rope, color="gray", alpha=0.3)
        ax_trust_params.set_yticks(range(plot_df.shape[0]))
        ax_trust_params.set_yticklabels(plot_df["Parameter"], ha="right", fontsize=small_fontsize)
        
        for j, (_, row) in enumerate(plot_df.iterrows()):
            ax_trust_params.text(
                row["Mean"] + xerr[1, j] + 0.02,
                j,
                f"{row['Mean']:.3f}",
                va="center",
                ha="left",
                fontsize=small_fontsize,
            )
        
        ax_trust_params.set_xlabel("Effect Size", fontsize=med_fontsize)
        ax_trust_params.set_xlim(-0.2, 0.8)
        ax_trust_params.spines["right"].set_visible(False)
        ax_trust_params.spines["top"].set_visible(False)
        ax_trust_params.tick_params(axis="both", labelsize=small_fontsize)
        ax_trust_params.set_title("Parameter Estimates", fontsize=big_fontsize, fontweight="bold")
    
    # Combined Private + Extremism parameters
    # Combine data from both private and extremism
    combined_data = []
    
    # Add private beliefs parameters
    private_summary = summary_dict['private']
    private_params = [
        "ground_truth",
        "time_ground_truth_researched", 
        "time_ground_truth_researched_istest",
    ]

    private_df = private_summary[private_summary["Parameter"].isin(private_params)].copy()
    if not private_df.empty:
        private_df["Variable"] = "Private"
        combined_data.append(private_df)
    
    # Add extremism parameters
    extremism_summary = summary_dict['extremism']
    extremism_params = [
        "ground_truth",
        "ground_truth_researched",
        "ground_truth_researched_istest",
    ]
    extremism_df = extremism_summary[extremism_summary["Parameter"].isin(extremism_params)].copy()
    if not extremism_df.empty:
        extremism_df["Variable"] = "Extremity"
        combined_data.append(extremism_df)
    
    # Combine and plot if we have data
    if combined_data:
        combined_plot_df = pd.concat(combined_data, ignore_index=True)
        
        # Create parameter labels with variable prefixes
        param_labels = []
        for _, row in combined_plot_df.iterrows():
            var = row["Variable"]
            param = row["Parameter"]
            
            if var == "Private":
                if param == "ground_truth":
                    label = "Progressive"
                elif param == "time_ground_truth_researched":
                    label = "Post x  Progressive × Researched"
                elif param == "time_ground_truth_researched_istest":
                    label = "Post x Progressive × Researched × ConvAI"
                else:
                    label = f"Private: {param}"
            elif var == "Extremity":
                if param == "ground_truth":
                    label = "Progressive"
                elif param == "ground_truth_researched":
                    label = "Progressive × Researched"
                elif param == "ground_truth_researched_istest":
                    label = "Progressive × Researched × ConvAI"
                else:
                    label = f"Extremity: {param}"
            else:
                label = f"{var}: {param}"
            
            param_labels.append(label)
        
        combined_plot_df["Parameter_Label"] = param_labels
        
        # Calculate error bars
        xerr = np.abs(combined_plot_df[["HDI_2.5%", "HDI_97.5%"]].T.values - combined_plot_df["Mean"].values)
        
        # Reverse order for plotting
        combined_plot_df = combined_plot_df.iloc[::-1].reset_index(drop=True)
        xerr = xerr[:, ::-1]
        param_labels = combined_plot_df["Parameter_Label"].tolist()
        
        # Create different colors for private vs extremism
        colors = []
        for _, row in combined_plot_df.iterrows():
            if row["Variable"] == "Private":
                colors.append("darkblue")
            else:  # Extremism
                colors.append("darkred")
        
        # Plot
        ax_combined_params.errorbar(
            combined_plot_df["Mean"],
            range(combined_plot_df.shape[0]),
            xerr=xerr,
            fmt="o",
            color="black",
            markersize=8,
            capsize=5,
        )
        
        # Color the markers differently
        for i, (_, row) in enumerate(combined_plot_df.iterrows()):
            ax_combined_params.scatter(
                row["Mean"],
                i,
                c=colors[i],
                s=64,
                zorder=3,
                edgecolors="white",
                linewidth=1
            )
        
        ax_combined_params.axvline(0, color="gray", linestyle="--", linewidth=0.5)
        ax_combined_params.axvspan(-rope, rope, color="gray", alpha=0.3)
        ax_combined_params.set_yticks(range(combined_plot_df.shape[0]))
        ax_combined_params.set_yticklabels(param_labels, ha="right", fontsize=small_fontsize)
        
        # Add effect size values
        for j, (_, row) in enumerate(combined_plot_df.iterrows()):
            ax_combined_params.text(
                row["Mean"] + xerr[1, j] + 0.02,
                j,
                f"{row['Mean']:.3f}",
                va="center",
                ha="left",
                fontsize=small_fontsize,
            )
        
        ax_combined_params.set_xlabel("Effect Size", fontsize=med_fontsize)
        # ax_combined_params.set_xlim(-0.2, 0.8)
        ax_combined_params.set_xlim(-0.5, 0.8)
        ax_combined_params.spines["right"].set_visible(False)
        ax_combined_params.spines["top"].set_visible(False)
        ax_combined_params.tick_params(axis="both", labelsize=small_fontsize)
        ax_combined_params.set_title("Parameter Estimates", fontsize=big_fontsize, fontweight="bold")
        
        # Add legend for the parameter colors
        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor='darkblue', label='Private Beliefs'),
            Patch(facecolor='darkred', label='Extremity')
        ]
        ax_combined_params.legend(handles=legend_elements, loc='upper left', fontsize=small_fontsize)
        
    else:
        # If no parameter data available
        ax_combined_params.text(0.5, 0.5, 'No parameter data available', 
                              ha='center', va='center', transform=ax_combined_params.transAxes,
                              fontsize=med_fontsize)
        ax_combined_params.set_xlim(0, 1)
        ax_combined_params.set_ylim(0, 1)
        ax_combined_params.set_xticks([])
        ax_combined_params.set_yticks([])
        for spine in ax_combined_params.spines.values():
            spine.set_visible(False)
    
    # Add overall legend with proper trust/distrust labeling
    from matplotlib.lines import Line2D
    
    # Create different legends for trust vs other variables
    truth_elements = [
        Line2D([0], [0], color=truth_colors[True], lw=6, label="Trust / Progressive Statements"),
        Line2D([0], [0], color=truth_colors[False], lw=6, label="Distrust / Conservative Statements"),
    ]
    
    control_elements = [
        Line2D([0], [0], color="black", lw=3, linestyle="-", label="Treatment (Researched)"),
        Line2D([0], [0], color="black", lw=3, linestyle=":", label="Control (Not Researched)"),
    ]
    
    # Add legends
    legend1 = fig.legend(
        handles=truth_elements,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.02),
        ncol=2,
        frameon=False,
        fontsize=med_fontsize,
    )
    legend2 = fig.legend(
        handles=control_elements,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.99),
        ncol=2,
        frameon=False,
        fontsize=med_fontsize,
    )

        # Save as pdf
    fig.savefig(
        "../figures/trust-private-beliefs-plot.pdf",
        bbox_inches="tight",
        format="pdf",
        dpi=300,
    )
    
    plt.tight_layout()
    return fig

# Usage example - same as before
df_dict = {
    'private': pd.read_csv("../data/Ztable_private_combined.csv"),
    'trust': pd.read_csv("../data/Ztable_trust_combined.csv"), 
    'extremism': pd.read_csv("../data/Ztable_extreme_combined.csv")
}

summary_dict = {
    'private': pd.read_csv("../parameter_estimates/summary_full_GLM_private_prompting_0_no_model.csv"),
    'trust': pd.read_csv("../parameter_estimates/summary_full_GLM_trust_prompting_0.csv"),
    'extremism': pd.read_csv("../parameter_estimates/summary_full_GLM_extreme_prompting_0.csv")
}

variable_names = {
    'trust': 'Trust', 
    'private': 'Private Beliefs',
    'extremism': 'Extremity'
}

# Process each dataframe
for var in df_dict:
    if not df_dict[var].empty:
        exclude_models = ["GPT4o_sycophancy_both", "GPT4o_persuasion"]
        df_dict[var] = df_dict[var][~df_dict[var]["model"].isin(exclude_models)]
        df_dict[var]["response"] = 8 - df_dict[var]["response"]
        print(f"Data for {var}: {len(df_dict[var])} rows")

# Call the function
fig = create_line_and_params_plot(
    df_dict=df_dict,
    summary_dict=summary_dict,
    ztable_dict=df_dict,
    variable_names=variable_names,
    truth_colors=truth_colors,
    line_styles=line_styles, 
    line_width=line_width,
    alpha_values=alpha_values,
    darker_truth_colors=darker_truth_colors,
    figsize=(20, 12)
)

plt.show()

# Plot sycophancy/persuasion prompting and LLMs separately

In [ ]:
def create_persuasion_sycophancy_plot(
    df_dict,  # Dictionary with 'persuasion': df, 'sycophancy': df
    variable_names,  # Dictionary with display names for each variable
    truth_colors,
    line_styles,
    line_width,
    alpha_values,
    figsize=(12, 6),
    big_fontsize=16,
    med_fontsize=14,
    small_fontsize=12,
):
    """
    Create a plot with line plots for persuasion and sycophancy
    
    Layout:
    - Single row: Persuasion line plot, Sycophancy line plot
    """
    
    # Set font properties to match the reference (serif font)
    plt.rcParams.update({
        'font.family': 'serif',
        'font.serif': ['Times New Roman', 'Times', 'DejaVu Serif'],
        'font.size': med_fontsize,
        'axes.labelsize': med_fontsize,
        'axes.titlesize': big_fontsize,
        'xtick.labelsize': small_fontsize,
        'ytick.labelsize': small_fontsize,
        'legend.fontsize': med_fontsize,
    })
    
    # Create figure with custom grid layout
    fig = plt.figure(figsize=figsize)
    
    # 1 row, 2 columns
    gs = fig.add_gridspec(
        1, 2,
        width_ratios=[1, 1],  # Equal width
        wspace=0.4,
    )
    
    # Only process persuasion and sycophancy for line plots
    variables = ['persuasion', 'sycophancy']
    comparison_values = {}
    
    # Model comparisons for each variable
    model_comparisons = {
        'persuasion': ['GPT4o_persuasion'],
        'sycophancy': ['GPT4o_sycophancy_both']
    }
    
    # Create subplots for persuasion and sycophancy line plots
    for i, var in enumerate(variables):
        # Line plots (single row)
        ax_line = fig.add_subplot(gs[0, i])
        
        # Get data for this variable
        df = df_dict[var]
        
        # Filter for only the models we want to compare
        models_to_keep = model_comparisons[var]
        df = df[df['model'].isin(models_to_keep)]
        
        # Debug: Print info about the data
        print(f"\n=== DEBUG INFO for {var} ===")
        print(f"DataFrame shape after filtering: {df.shape}")
        print(f"Models included: {df['model'].unique()}")
        print(f"DataFrame empty: {df.empty}")
        if not df.empty:
            print(f"Unique values in 'iscontrol': {df['iscontrol'].unique()}")
            print(f"Unique values in 'researched': {df['researched'].unique()}")
            print(f"Unique values in 'ground_truth': {df['ground_truth'].unique()}")
            print(f"Unique values in 'presented': {df['presented'].unique()}")
            print(f"Response range: {df['response'].min()} to {df['response'].max()}")
            print(f"Response mean: {df['response'].mean():.2f}")
        
        # Check if we have data for this variable
        if df.empty:
            ax_line.text(0.5, 0.5, f'No data available\nfor {variable_names[var]}', 
                       ha='center', va='center', transform=ax_line.transAxes,
                       fontsize=med_fontsize)
            ax_line.set_xlim(0, 1)
            ax_line.set_ylim(0, 1)
            ax_line.set_xticks([])
            ax_line.set_yticks([])
            for spine in ax_line.spines.values():
                spine.set_visible(False)
            continue
        
        # Initialize comparison values for this variable
        comparison_values[var] = {"true": {"pre": {}, "post": {}}, "false": {"pre": {}, "post": {}}}
        
        # # Track if we actually plot any data
        # data_plotted = False
        
        # # Process data for line plots - only using conversational AI data (iscontrol == 1 or "test")
        # df_conv = df[(df["iscontrol"] == 1) | (df["iscontrol"] == "test")]
        
        # if df_conv.empty:
        #     ax_line.text(0.5, 0.5, f'No conversational AI data\nfor {variable_names[var]}', 
        #                ha='center', va='center', transform=ax_line.transAxes,
        #                fontsize=med_fontsize)
        #     continue
        
        # N = df_conv["subject"].nunique()

        # Calculate N from the full filtered dataset (before restricting to conversational AI)
        # N = df["subject"].nunique()
        # Each model has its own set of subjects, so we need to count unique model_subject combinations
        df['model_subject'] = df['model'] + '_' + df['subject'].astype(str)
        N = df["model_subject"].nunique()
        
        # Track if we actually plot any data
        data_plotted = False
        
        # Process data for line plots - only using conversational AI data (iscontrol == 1 or "test")
        df_conv = df[(df["iscontrol"] == 1) | (df["iscontrol"] == "test")]
        
        if df_conv.empty:
            ax_line.text(0.5, 0.5, f'No conversational AI data\nfor {variable_names[var]}', 
                       ha='center', va='center', transform=ax_line.transAxes,
                       fontsize=med_fontsize)
            continue
        
        for researched in df_conv["researched"].unique():
            df_researched = df_conv[df_conv["researched"] == researched]
            
            # Plot lines for each truth value
            for groundtruth in df_researched["ground_truth"].unique():
                df_groundtruth = df_researched[df_researched["ground_truth"] == groundtruth]
                
                pre_responses = df_groundtruth[df_groundtruth["presented"] == "pre"]
                post_responses = df_groundtruth[df_groundtruth["presented"] == "post"]
                
                print(f"  Plotting {var}-{researched}-{groundtruth}: pre={len(pre_responses)}, post={len(post_responses)}")
                
                if pre_responses.empty or post_responses.empty:
                    print(f"    Skipping - empty data")
                    continue
                
                # Use colors based on truth value
                color = truth_colors[bool(groundtruth)]
                
                # Calculate means
                mean_pre = pre_responses["response"].mean()
                mean_post = post_responses["response"].mean()
                
                print(f"    Means: pre={mean_pre:.2f}, post={mean_post:.2f}")
                
                # Store values for comparison
                truth_key = "true" if bool(groundtruth) else "false"
                comparison_values[var][truth_key]["pre"][researched] = mean_pre
                comparison_values[var][truth_key]["post"][researched] = mean_post
                
                # Plot line using numeric positions (0 for Pre, 1 for Post)
                line = ax_line.plot(
                    [0, 1],  # Use numeric positions instead of strings
                    [mean_pre, mean_post],
                    marker="o",
                    markersize=10,
                    markeredgecolor="white",
                    markeredgewidth=1.5,
                    color=color,
                    linestyle=line_styles[researched],
                    linewidth=line_width[researched],
                    alpha=alpha_values[researched],
                )
                
                print(f"    Plotted line successfully")
                
                # Calculate and add error bars
                pre_std = pre_responses["response"].std()
                post_std = post_responses["response"].std()
                pre_n = pre_responses["response"].count()
                post_n = post_responses["response"].count()
                pre_error = 1.96 * (pre_std / np.sqrt(pre_n))
                post_error = 1.96 * (post_std / np.sqrt(post_n))
                
                ax_line.errorbar(0, mean_pre, yerr=pre_error, fmt="none", ecolor=color, 
                           capsize=5, elinewidth=1.5, alpha=alpha_values[researched])
                ax_line.errorbar(1, mean_post, yerr=post_error, fmt="none", ecolor=color,
                           capsize=5, elinewidth=1.5, alpha=alpha_values[researched])
                
                data_plotted = True
        
        print(f"Data plotted for {var}: {data_plotted}")
        
        # Format line plot axis
        # Grid and axis settings
        ax_line.xaxis.grid(False)
        ax_line.yaxis.grid(True, linestyle="--", alpha=0.7)
        ax_line.set_xlim(-0.35, 1.35)
        
        # Set fixed y-limits for better zoom
        ax_line.set_ylim(3.6, 4.8)
        
        # Format axes using numeric positions
        ax_line.set_xticks([0, 1])  # Use numeric positions
        ax_line.set_xticklabels(["Pre", "Post"], fontsize=med_fontsize, fontweight="bold")
        
        # Style spines
        ax_line.spines["right"].set_visible(False)
        ax_line.spines["top"].set_visible(False)
        ax_line.spines["bottom"].set_visible(False)
        ax_line.spines["bottom"].set_color("gray")
        ax_line.spines["left"].set_color("gray")
        ax_line.spines["bottom"].set_linewidth(0.5)
        ax_line.spines["left"].set_linewidth(0.5)
        ax_line.tick_params(axis="y", labelsize=med_fontsize)
        
        # Add midpoint line
        ax_line.axhline(4, color="black", linestyle="-", linewidth=1.5, alpha=0.7)
        
        # Add y-axis label only to the leftmost plot
        if i == 0:
            ax_line.set_ylabel(
                "Mean Likert\n(1=disagree,\n7=agree)",
                fontsize=med_fontsize,
                labelpad=70,
                rotation=0,
                ha="center",
            )
                
        # Add variable name above the plot
        # fig.text(0.25 + i*0.5, 0.95, variable_names[var], fontsize=big_fontsize+2, 
        #         fontweight="bold", ha="center", transform=fig.transFigure)
        # fig.text(0.25 + i*0.5, 0.95, f"{variable_names[var]} (N = {N})", fontsize=big_fontsize+2, 
        #         fontweight="bold", ha="center", transform=fig.transFigure)
        if i == 0:  # Left label (Persuasion)
            fig.text(0.30, 0.95, f"{variable_names[var]} (N = {N})", fontsize=big_fontsize+2, 
                    fontweight="bold", ha="center", transform=fig.transFigure)
        else:  # Right label (Sycophancy)
            fig.text(0.75, 0.95, f"{variable_names[var]} (N = {N})", fontsize=big_fontsize+2, 
                    fontweight="bold", ha="center", transform=fig.transFigure)


        # Add panel labels
        panel_label = "A" if i == 0 else "B"
        ax_line.annotate(
            panel_label,
            xy=(-0.3, 1.11),
            xycoords="axes fraction",
            fontsize=25,
            fontweight="bold",
            bbox=dict(boxstyle="circle", fc="white", ec="black", alpha=0.8),
        )
    
    # Add overall legend 
    from matplotlib.lines import Line2D
    
    # Create legend for true/false information
    truth_elements = [
        Line2D([0], [0], color=truth_colors[True], lw=6, label="True Information"),
        Line2D([0], [0], color=truth_colors[False], lw=6, label="False Information"),
    ]
    
    control_elements = [
        Line2D([0], [0], color="black", lw=3, linestyle="-", label="Treatment (Researched)"),
        Line2D([0], [0], color="black", lw=3, linestyle=":", label="Control (Not Researched)"),
    ]
    
    # Add legends
    legend1 = fig.legend(
        handles=truth_elements,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.17),  
        ncol=2,
        frameon=False,
        fontsize=med_fontsize,
    )
    legend2 = fig.legend(
        handles=control_elements,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.13),  
        ncol=2,
        frameon=False,
        fontsize=med_fontsize,
    )
    
    plt.tight_layout()
    
    # Save as pdf
    fig.savefig(
        "../figures/persuasion-sycophancy-plot.pdf",
        bbox_inches="tight",
        format="pdf",
        dpi=300,
    )
    
    return fig

# Usage example:
df_dict = {
    'persuasion': pd.read_csv("../data/Ztable_misinfo_combined.csv"),
    'sycophancy': pd.read_csv("../data/Ztable_misinfo_combined.csv"),
}

variable_names = {
    'persuasion': 'Persuasion',
    'sycophancy': 'Sycophancy'
}

# Process each dataframe
for var in df_dict:
    if not df_dict[var].empty:
        df_dict[var]["response"] = 8 - df_dict[var]["response"]
        print(f"Data for {var}: {len(df_dict[var])} rows")

# Call the function (no need for summary_dict anymore)
fig = create_persuasion_sycophancy_plot(
    df_dict=df_dict,
    variable_names=variable_names,
    truth_colors=truth_colors,
    line_styles=line_styles, 
    line_width=line_width,
    alpha_values=alpha_values,
    figsize=(12, 6)
)

plt.show()

In [ ]:
tmp_df = df_dict["persuasion"]
tmp_df

In [ ]:
def create_persuasion_sycophancy_plot(
    df_dict,  # Dictionary with 'persuasion': df, 'sycophancy': df
    variable_names,  # Dictionary with display names for each variable
    truth_colors,
    line_styles,
    line_width,
    alpha_values,
    figsize=(24, 6),  # Made wider to accommodate 5 plots
    big_fontsize=16,
    med_fontsize=14,
    small_fontsize=12,
):
    """
    Create a plot with line plots for persuasion, sycophancy, and 3 additional models
    
    Layout:
    - Single row: Persuasion, Sycophancy, GPT-4o, Claude, Mistral
    """
    
    # Set font properties to match the reference (serif font)
    plt.rcParams.update({
        'font.family': 'serif',
        'font.serif': ['Times New Roman', 'Times', 'DejaVu Serif'],
        'font.size': med_fontsize,
        'axes.labelsize': med_fontsize,
        'axes.titlesize': big_fontsize,
        'xtick.labelsize': small_fontsize,
        'ytick.labelsize': small_fontsize,
        'legend.fontsize': med_fontsize,
    })
    
    # Create figure with custom grid layout
    fig = plt.figure(figsize=figsize)
    
    # 1 row, 5 columns
    gs = fig.add_gridspec(
        1, 5,
        width_ratios=[1, 1, 1, 1, 1],  # Equal width
        wspace=0.8,  # Increased from 0.3
    )
    
    # Define all variables/models to plot
    plot_configs = [
        ('persuasion', 'GPT4o_persuasion', 'Persuasion'),
        ('sycophancy', 'GPT4o_sycophancy_both', 'Sycophancy'), 
        ('gpt4o', 'GPT4o', 'GPT-4o'),
        ('claude', 'claude', 'Claude'),
        ('mistral', 'mistral', 'Mistral')
    ]
    
    # Create subplots for all models
    for i, (var_key, model_name, display_name) in enumerate(plot_configs):
        # Line plots (single row)
        ax_line = fig.add_subplot(gs[0, i])
        
        # # Get data - use misinfo data for the last 3 models
        # if var_key in ['persuasion', 'sycophancy']:
        #     df = df_dict[var_key].copy()
        # else:
        #     # For the additional models, use the persuasion dataset but filter for different models
        #     df = df_dict['persuasion'].copy()
        
        # # Filter for the specific model
        # df = df[df['model'] == model_name]

        # Get data - use misinfo data for the last 3 models
        if var_key in ['persuasion', 'sycophancy']:
            df = df_dict[var_key].copy()
        else:
            # For the additional models, use the persuasion dataset but filter for different models
            df = df_dict['persuasion'].copy()
        
        # Filter for the specific model
        df = df[df['model'] == model_name]
        
        # Debug: Print info about the data
        print(f"\n=== DEBUG INFO for {display_name} ({model_name}) ===")
        print(f"DataFrame shape after filtering: {df.shape}")
        print(f"Models included: {df['model'].unique()}")
        print(f"DataFrame empty: {df.empty}")
        if not df.empty:
            print(f"Unique values in 'iscontrol': {df['iscontrol'].unique()}")
            print(f"Unique values in 'researched': {df['researched'].unique()}")
            print(f"Unique values in 'ground_truth': {df['ground_truth'].unique()}")
            print(f"Unique values in 'presented': {df['presented'].unique()}")
            print(f"Response range: {df['response'].min()} to {df['response'].max()}")
            print(f"Response mean: {df['response'].mean():.2f}")
        
        # Check if we have data for this variable
        if df.empty:
            ax_line.text(0.5, 0.5, f'No data available\nfor {display_name}', 
                       ha='center', va='center', transform=ax_line.transAxes,
                       fontsize=med_fontsize)
            ax_line.set_xlim(0, 1)
            ax_line.set_ylim(0, 1)
            ax_line.set_xticks([])
            ax_line.set_yticks([])
            for spine in ax_line.spines.values():
                spine.set_visible(False)
            continue
        
        # Calculate N from the full filtered dataset
        N = df["subject"].nunique()
        
        # Track if we actually plot any data
        data_plotted = False
        
        # Process data for line plots - only using conversational AI data (iscontrol == 1 or "test")
        df_conv = df[(df["iscontrol"] == 1) | (df["iscontrol"] == "test")]
        
        # Calculate N from the full filtered dataset
        # N = df_conv["subject"].nunique()
        # Each model has its own set of subjects, so we need to count unique model_subject combinations
        df_conv['model_subject'] = df_conv['model'] + '_' + df_conv['subject'].astype(str)
        N = df_conv["model_subject"].nunique()

        if df_conv.empty:
            ax_line.text(0.5, 0.5, f'No conversational AI data\nfor {display_name}', 
                       ha='center', va='center', transform=ax_line.transAxes,
                       fontsize=med_fontsize)
            continue
        
        for researched in df_conv["researched"].unique():
            df_researched = df_conv[df_conv["researched"] == researched]
            
            # Plot lines for each truth value
            for groundtruth in df_researched["ground_truth"].unique():
                df_groundtruth = df_researched[df_researched["ground_truth"] == groundtruth]
                
                pre_responses = df_groundtruth[df_groundtruth["presented"] == "pre"]
                post_responses = df_groundtruth[df_groundtruth["presented"] == "post"]
                
                print(f"  Plotting {display_name}-{researched}-{groundtruth}: pre={len(pre_responses)}, post={len(post_responses)}")
                
                if pre_responses.empty or post_responses.empty:
                    print(f"    Skipping - empty data")
                    continue
                
                # Use colors based on truth value
                color = truth_colors[bool(groundtruth)]
                
                # Calculate means
                mean_pre = pre_responses["response"].mean()
                mean_post = post_responses["response"].mean()
                
                print(f"    Means: pre={mean_pre:.2f}, post={mean_post:.2f}")
                
                # Plot line using numeric positions (0 for Pre, 1 for Post)
                line = ax_line.plot(
                    [0, 1],  # Use numeric positions instead of strings
                    [mean_pre, mean_post],
                    marker="o",
                    markersize=10,
                    markeredgecolor="white",
                    markeredgewidth=1.5,
                    color=color,
                    linestyle=line_styles[researched],
                    linewidth=line_width[researched],
                    alpha=alpha_values[researched],
                )
                
                print(f"    Plotted line successfully")
                
                # Calculate and add error bars
                pre_std = pre_responses["response"].std()
                post_std = post_responses["response"].std()
                pre_n = pre_responses["response"].count()
                post_n = post_responses["response"].count()
                pre_error = 1.96 * (pre_std / np.sqrt(pre_n))
                post_error = 1.96 * (post_std / np.sqrt(post_n))
                
                ax_line.errorbar(0, mean_pre, yerr=pre_error, fmt="none", ecolor=color, 
                           capsize=5, elinewidth=1.5, alpha=alpha_values[researched])
                ax_line.errorbar(1, mean_post, yerr=post_error, fmt="none", ecolor=color,
                           capsize=5, elinewidth=1.5, alpha=alpha_values[researched])
                
                data_plotted = True
        
        print(f"Data plotted for {display_name}: {data_plotted}")
        
        # Format line plot axis
        ax_line.xaxis.grid(False)
        ax_line.yaxis.grid(True, linestyle="--", alpha=0.7)
        ax_line.set_xlim(-0.35, 1.35)
        ax_line.set_ylim(3.4, 4.8)
        
        # Format axes using numeric positions
        ax_line.set_xticks([0, 1])
        ax_line.set_xticklabels(["Pre", "Post"], fontsize=med_fontsize, fontweight="bold")
        
        # Style spines
        ax_line.spines["right"].set_visible(False)
        ax_line.spines["top"].set_visible(False)
        ax_line.spines["bottom"].set_visible(False)
        ax_line.spines["bottom"].set_color("gray")
        ax_line.spines["left"].set_color("gray")
        ax_line.spines["bottom"].set_linewidth(0.5)
        ax_line.spines["left"].set_linewidth(0.5)
        ax_line.tick_params(axis="y", labelsize=med_fontsize)
        
        # Add midpoint line
        ax_line.axhline(4, color="black", linestyle="-", linewidth=1.5, alpha=0.7)
        
        # Add y-axis label only to the leftmost plot
        if i == 0:
            ax_line.set_ylabel(
                "Mean Likert\n(1=disagree,\n7=agree)",
                fontsize=med_fontsize,
                labelpad=70,
                rotation=0,
                ha="center",
            )
        
        # Add variable name above the plot
        x_positions = [0.17, 0.34, 0.51, 0.68, 0.85] 
        fig.text(x_positions[i], 0.91, f"{display_name}\n(N = {N})", fontsize=big_fontsize+2, 
                fontweight="bold", ha="center", transform=fig.transFigure)
        
        # Add panel labels
        panel_labels = ["A", "B", "C", "D", "E"]
        ax_line.annotate(
            panel_labels[i],
            xy=(-0.4, 1.08),
            xycoords="axes fraction",
            fontsize=25,
            fontweight="bold",
            bbox=dict(boxstyle="circle", fc="white", ec="black", alpha=0.8),
        )
    
    # Add overall legend 
    from matplotlib.lines import Line2D
    
    # Create legend for true/false information
    truth_elements = [
        Line2D([0], [0], color=truth_colors[True], lw=6, label="True Information"),
        Line2D([0], [0], color=truth_colors[False], lw=6, label="False Information"),
    ]
    
    control_elements = [
        Line2D([0], [0], color="black", lw=3, linestyle="-", label="Treatment (Researched)"),
        Line2D([0], [0], color="black", lw=3, linestyle=":", label="Control (Not Researched)"),
    ]
    
    # Add legends
    legend1 = fig.legend(
        handles=truth_elements,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.14),
        ncol=2,
        frameon=False,
        fontsize=med_fontsize,
    )
    legend2 = fig.legend(
        handles=control_elements,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.10),
        ncol=2,
        frameon=False,
        fontsize=med_fontsize,
    )
    
    plt.tight_layout()
    
    # Save as pdf
    fig.savefig(
        "../figures/persuasion-sycophancy-models-plot.pdf",
        bbox_inches="tight",
        format="pdf",
        dpi=300,
    )
    
    return fig

# Usage example - same as before
fig = create_persuasion_sycophancy_plot(
    df_dict=df_dict,
    variable_names=variable_names,
    truth_colors=truth_colors,
    line_styles=line_styles, 
    line_width=line_width,
    alpha_values=alpha_values,
    figsize=(20, 6)  # Made wider for 5 plots
)

plt.show()